# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chisman001/ML-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

This is a ranking task because the goal is to prioritize content pages for editorial review, rather than simply classify every page as good or bad. I will use Logistic Regression as the first learned model because it is simple, interpretable, and provides probability scores that can be used to rank pages. I will evaluate the ranking using Precision@50, which matches the metric selected in the previous week.

I will compare the learned model against the Week-4 rule-based baseline using the same test data and the same metric. This keeps the comparison fair and makes it possible to determine whether the model provides useful improvement over the simpler baseline.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

RANDOM_STATE = 42

MODEL_FEATURES = [
    "total_impressions",
    "total_clicks",
    "ctr_pct",
    "avg_position",
    "days_since_update",
    "word_count",
]

print("Random state:", RANDOM_STATE)
print("Planned features:")
for feature in MODEL_FEATURES:
    print("-", feature)



Random state: 42
Planned features:
- total_impressions
- total_clicks
- ctr_pct
- avg_position
- days_since_update
- word_count


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a client-grouped train/test split, holding out approximately 20% of clients for testing. Pages from the same client can share website structure, content patterns, and traffic behavior, so randomly splitting individual pages could allow information from the same client to appear in both training and testing data.

The model will use information available through March 2026 as its features. The outcome label will be calculated from the future April–May 2026 period. This prevents future information from being used to predict the past.

I will use the same held-out test clients for both the learned model and the Week-4 baseline.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "duckdb", "pandas", "numpy", "scikit-learn"],
        check=True,
    )
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

if HF_TOKEN:
    con.execute(
        f"""
        CREATE OR REPLACE SECRET hf_secret (
            TYPE HUGGINGFACE,
            TOKEN '{HF_TOKEN}'
        );
        """
    )
    print("Hugging Face authentication configured.")
else:
    print("HF_TOKEN not set — set Colab secret or HF_TOKEN env var before warehouse queries.")

print("DuckDB ready.")


Hugging Face authentication configured.
DuckDB ready.


In [14]:
# Warehouse paths
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
april_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet"
may_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/data_0.parquet"

content_path = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

print("March:", con.execute(f"SELECT COUNT(*) FROM '{march_path}'").fetchone()[0])
print("April:", con.execute(f"SELECT COUNT(*) FROM '{april_path}'").fetchone()[0])
print("May:", con.execute(f"SELECT COUNT(*) FROM '{may_path}'").fetchone()[0])

# Check the March warehouse schema before building the modeling dataset

con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").show()


March: 9841378
April: 10424730
May: 11687376
┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT  

In [15]:
# Check the content dimension schema

con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{content_path}')
""").show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [16]:
decision_date = "2026-03-31"

# March page-level features (information available at the decision date)
march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_sum_position) AS sum_position,
        COUNT(DISTINCT report_date) AS days_observed,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE 0
        END AS ctr_pct,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0
    GROUP BY client_hash_id, content_hash_id
""").df()

print("March feature rows:", len(march_features))
march_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March feature rows: 175304


,client_hash_id,content_hash_id,total_impressions,total_clicks,sum_position,days_observed,ctr_pct,avg_position
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,70.0,0.0,332.0,20,0.000000,4.742857
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,87333.0,31,0.202784,8.049866
2,client_62f4a7e64f5e0096,content_e689bc511192751a,56.0,0.0,359.0,23,0.000000,6.410714
3,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,47.0,0.0,718.0,19,0.000000,15.276596
4,client_62f4a7e64f5e0096,content_df22bda1218f13ff,2099.0,1.0,5525.0,31,0.047642,2.632206


In [17]:
content = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        word_count,
        content_type,
        is_published,
        is_deleted
    FROM read_parquet('{content_path}')
    WHERE content_updated_date IS NOT NULL
      AND content_updated_date <= DATE '{decision_date}'
      AND is_deleted IS NOT TRUE
""").df()

content["content_updated_date"] = pd.to_datetime(content["content_updated_date"])
decision_ts = pd.Timestamp(decision_date)
content["days_since_update"] = (decision_ts - content["content_updated_date"]).dt.days

features = march_features.merge(
    content,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

print("Feature rows after content merge:", features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows after content merge: (27684, 14)


,client_hash_id,content_hash_id,total_impressions,total_clicks,sum_position,days_observed,ctr_pct,avg_position,content_updated_date,word_count,content_type,is_published,is_deleted,days_since_update
0,client_8dbf3abdf07569e0,content_06ca04dd3afd1820,107.0,0.0,2125.0,30,0.0,19.859813,2026-02-25,1878,keyword article,True,False,34
1,client_8dbf3abdf07569e0,content_d5d351913e3ff832,39.0,0.0,2873.0,17,0.0,73.666667,2026-02-25,2101,keyword article,True,False,34
2,client_8dbf3abdf07569e0,content_121e3f7e8a310107,25.0,1.0,191.0,15,4.0,7.640000,2026-02-25,1458,keyword article,True,False,34
3,client_8dbf3abdf07569e0,content_94a216edbce6a6b7,6.0,0.0,331.0,5,0.0,55.166667,2026-02-25,1689,keyword article,True,False,34
4,client_8dbf3abdf07569e0,content_0229eec19f724ef9,3.0,0.0,28.0,3,0.0,9.333333,2026-02-25,1756,keyword article,True,False,34


In [18]:
# Future April–May performance for the outcome label (not used as model features)
future_performance = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_impressions,
        SUM(gsc_clicks) AS future_clicks,
        COUNT(DISTINCT report_date) AS future_days_observed
    FROM (
        SELECT * FROM read_parquet('{april_path}')
        UNION ALL
        SELECT * FROM read_parquet('{may_path}')
    )
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
""").df()

features["march_monthly_impressions"] = (
    features["total_impressions"] / features["days_observed"].clip(lower=1) * 30.5
)

future_performance["future_monthly_impressions"] = (
    future_performance["future_impressions"]
    / future_performance["future_days_observed"].clip(lower=1)
    * 30.5
)

features = features.merge(
    future_performance[
        [
            "client_hash_id",
            "content_hash_id",
            "future_monthly_impressions",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

features["future_impression_change_pct"] = (
    (features["future_monthly_impressions"] - features["march_monthly_impressions"])
    / features["march_monthly_impressions"].replace(0, np.nan)
) * 100

# Modeling definition: >=100 March monthly impressions and >=30% decline in April–May
features["refresh_label"] = (
    (features["march_monthly_impressions"] >= 100)
    & (features["future_impression_change_pct"] <= -30)
).astype(int)

print("Rows:", len(features))
print("Positive labels:", features["refresh_label"].sum())
print("Positive rate:", round(features["refresh_label"].mean(), 4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 25665
Positive labels: 10680
Positive rate: 0.4161


In [19]:
# Week-4 baseline score (same rule as w04_baseline_score.ipynb)
baseline_scores = con.sql(f"""
    WITH daily AS (
        SELECT
            p.client_hash_id,
            p.content_hash_id,
            p.gsc_impressions,
            p.gsc_clicks,
            p.gsc_avg_position
        FROM read_parquet('{march_path}') p
        WHERE p.gsc_data_available IS TRUE
          AND p.gsc_impressions > 0
          AND p.gsc_avg_position > 0
    ),
    page_month AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM daily
        GROUP BY client_hash_id, content_hash_id
    ),
    base AS (
        SELECT
            p.client_hash_id,
            p.content_hash_id,
            p.total_impressions,
            p.total_clicks,
            CASE
                WHEN p.total_impressions > 0
                THEN p.total_clicks * 100.0 / p.total_impressions
                ELSE NULL
            END AS ctr_pct,
            p.avg_position,
            date_diff('day', c.content_updated_date, DATE '{decision_date}') AS days_since_update
        FROM page_month p
        JOIN read_parquet('{content_path}') c
          ON p.client_hash_id = c.client_hash_id
         AND p.content_hash_id = c.content_hash_id
        WHERE c.content_updated_date IS NOT NULL
          AND c.content_updated_date <= DATE '{decision_date}'
    ),
    scored AS (
        SELECT
            *,
            CASE
                WHEN total_impressions >= 100 AND avg_position <= 10 AND ctr_pct < 0.20 THEN 70
                WHEN total_impressions >= 30 AND avg_position <= 10 AND ctr_pct < 0.20 THEN 50
                WHEN total_impressions >= 30 AND avg_position <= 20 AND ctr_pct < 0.15 THEN 40
                ELSE 0
            END AS ctr_score,
            CASE
                WHEN days_since_update >= 365 THEN 30
                WHEN days_since_update >= 180 THEN 20
                WHEN days_since_update >= 90 THEN 10
                ELSE 0
            END AS stale_score
        FROM base
    )
    SELECT
        client_hash_id,
        content_hash_id,
        ctr_score + stale_score AS baseline_score
    FROM scored
""").df()

features = features.merge(
    baseline_scores,
    on=["client_hash_id", "content_hash_id"],
    how="left",
)
features["baseline_score"] = features["baseline_score"].fillna(0)

model_features = MODEL_FEATURES.copy()
print("Modeling rows with baseline score:", len(features))
features[model_features + ["refresh_label", "baseline_score"]].head()

Modeling rows with baseline score: 25665


,total_impressions,total_clicks,ctr_pct,avg_position,days_since_update,word_count,refresh_label,baseline_score
0,107.0,0.0,0.0,19.859813,34,1878,0,40
1,39.0,0.0,0.0,73.666667,34,2101,0,0
2,6.0,0.0,0.0,55.166667,34,1689,0,0
3,142.0,0.0,0.0,2.563380,34,<NA>,1,70
4,109.0,0.0,0.0,74.238532,34,<NA>,0,0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


### Comparison approach

Both the Week-4 baseline and the Logistic Regression model are evaluated on the same client-held-out test pages using Precision@50. The base rate shows how often positive labels appear in the test set at random.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def precision_at_k(y_true, scores, k=50):
    frame = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


X = features[model_features].copy()
y = features["refresh_label"].copy()
groups = features["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())
print("Train label rate:", round(y_train.mean(), 4))
print("Test label rate:", round(y_test.mean(), 4))

model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                class_weight="balanced",
            ),
        ),
    ]
)

model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]
baseline_test_scores = features.iloc[test_idx]["baseline_score"].to_numpy()

comparison = pd.DataFrame(
    [
        {
            "method": "Base rate",
            "precision_at_50": round(float(y_test.mean()), 4),
            "notes": "Share of positives in the test set",
        },
        {
            "method": "Week-4 baseline",
            "precision_at_50": precision_at_k(y_test, baseline_test_scores, 50),
            "notes": "Rule-based CTR + staleness score",
        },
        {
            "method": "Logistic Regression",
            "precision_at_50": precision_at_k(y_test, model_scores, 50),
            "notes": "Learned model probabilities",
        },
    ]
)

comparison["precision_at_50"] = comparison["precision_at_50"].round(4)
comparison



Train rows: 23377
Test rows: 2288
Train clients: 23
Test clients: 6
Train label rate: 0.4203
Test label rate: 0.3733


,method,precision_at_50,notes
0,Base rate,0.3733,Share of positives in the test set
1,Week-4 baseline,0.8600,Rule-based CTR + staleness score
2,Logistic Regression,0.6600,Learned model probabilities


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model is evaluated as decision support, not as proof that a refresh will improve performance. The observed errors below come from the client-held-out test set only.

On the held-out test clients, the Week-4 baseline achieved higher Precision@50 (0.86) than Logistic Regression (0.66). Both improved over the base rate (0.37). The baseline's strength suggests that low-CTR and staleness signals align with observed future impression declines in this slice. The model still ranks above random but did not outperform the hand-written rule on this metric. These numbers are measured on this client holdout only and should be treated as directional decision support, not a guarantee on unseen clients.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

test_frame = features.iloc[test_idx].copy()
test_frame["model_score"] = model_scores

# Where the model ranks highest but the future label is 0
false_positives = (
    test_frame.sort_values("model_score", ascending=False)
    .head(50)
    .query("refresh_label == 0")
    [[
        "total_impressions",
        "ctr_pct",
        "avg_position",
        "days_since_update",
        "word_count",
        "future_impression_change_pct",
        "model_score",
    ]]
)

print("False positives in model top 50:", len(false_positives))
false_positives.head(3)

coefficients = pd.DataFrame(
    {
        "feature": model_features,
        "coefficient": model.named_steps["classifier"].coef_[0],
    }
).sort_values("coefficient", key=abs, ascending=False)

print("\nTop logistic regression coefficients:")
coefficients



False positives in model top 50: 17

Top logistic regression coefficients:


,feature,coefficient
2,ctr_pct,-0.660693
0,total_impressions,0.477365
3,avg_position,-0.412975
1,total_clicks,-0.388116
4,days_since_update,-0.384936
5,word_count,0.308143


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.